# prep.tableS1

This table converts the McLeish et al. 2024 table S1 and S2 into a single Table that will be labelled as Table S1 in the article, aiming to provide:

- Library names
- Library site locations
- Library number of extracts
- Library host taxon

Among others. 


In [2]:
import pandas as pd
import seaborn as sns 
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
db.toc()

┌────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      name      │                                                                                  description                                                                                   │
│    varchar     │                                                                                    varchar                                                                                     │
├────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_bacteriaHits │ This table contains all the MOTUS hits obtained, regardless of their status. It contains the library where the detection took place, taxid, scientific name and the PAB labels │
│ D_PABHits      │ T

## Load libraries and sampling tables (S1 and S2)

In [3]:
libraries = pd.read_csv("input/mcleish24.TableS2.csv", sep=';')
sampling = pd.read_csv("input/mcleish24.TableS1.csv", sep=';')
sampling = sampling.groupby(['Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude'], as_index = False)['Date'].apply(list)
sampling['Date'] = sampling['Date'].apply(lambda x: '; '.join(x))

In [4]:
sampling

,Site_code,Collection_code,Location,Longitude,Latitude,Date
0,C1,C1F,Aranjuez,-3.593308,40.051302,1/2/16
1,C1,C3F,Aranjuez,-3.593308,40.051302,30/1/17
2,C2,C2F,Aranjuez,-3.599064,40.043193,1/2/16
3,C2,C4F,Aranjuez,-3.599064,40.043193,30/1/17
4,E1,E1F,Aranjuez,-3.500323,40.059138,19/11/15; 10/11/16
5,E1,E1P,Aranjuez,-3.500323,40.059138,23/5/16; 3/5/17
6,E2,E2F,San Martín de la Vega,-3.536191,40.234966,19/11/15; 10/11/16
7,E2,E2P,San Martín de la Vega,-3.536191,40.234966,23/5/16; 10/5/17
8,E3,E3F,Ambite,-3.196057,40.089637,1/2/16; 10/11/16
9,E3,E3P,Ambite,-3.196057,40.089637,31/5/16; 3/5/17


In [5]:
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


## Merge

The merge process requires dealing with libraries that were pooled from various sites. 

In [6]:
libraries['Collection_code'] = libraries['Collection_code'].apply(lambda x: x.split("_"))
tmp = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'Collection_code']].to_dict(orient='records')
library_collection_code = []
for item in tmp:
    for collection_code in item['Collection_code']:
        library_collection_code.append({"Library_code": item['Library_code'], "Collection_code": collection_code})
library_collection_code = pd.DataFrame.from_records(library_collection_code)
library_collection_code

,Library_code,Collection_code
0,PV001,M1V
1,PV002,M1V
2,PV003,M1V
3,PV003bgi,M1V
4,PV004bgi,M1V
...,...,...
330,PV586,H1P
331,PV587,H1P
332,PV588,H1P
333,PV589,H1P


In [7]:
library_site = pd.merge(library_collection_code, sampling, on='Collection_code') # type: ignore
library_site


,Library_code,Collection_code,Site_code,Location,Longitude,Latitude,Date
0,PV001,M1V,M1,Aranjuez,-3.345220,40.031840,16/7/15
1,PV002,M1V,M1,Aranjuez,-3.345220,40.031840,16/7/15
2,PV003,M1V,M1,Aranjuez,-3.345220,40.031840,16/7/15
3,PV003bgi,M1V,M1,Aranjuez,-3.345220,40.031840,16/7/15
4,PV004bgi,M1V,M1,Aranjuez,-3.345220,40.031840,16/7/15
...,...,...,...,...,...,...,...
330,PV586,H1P,H1,Villaconejos,-3.477574,40.049330,21/4/16
331,PV587,H1P,H1,Villaconejos,-3.477574,40.049330,21/4/16
332,PV588,H1P,H1,Villaconejos,-3.477574,40.049330,21/4/16
333,PV589,H1P,H1,Villaconejos,-3.477574,40.049330,21/4/16


In [8]:
library_site.value_counts('Library_code')

Library_code
PV578    3
PV570    3
PV582    2
PV566    2
PV562    2
        ..
PV115    1
PV114    1
PV113    1
PV112    1
PV590    1
Name: count, Length: 323, dtype: int64

In [9]:
sites = pd.merge(
    library_site,
    libraries[['Library_code', 'Habitat', 'No_extracts', 'Host_taxon']], on='Library_code', how='left'
)
# sites['Habitat_type'] = sites['Habitat'].map(sites_to_type)
# sites = sites.drop_duplicates(subset=['Site_code'])[['Site_code', 'Longitude', 'Latitude', 'Location', 'Habitat', 'Habitat_type']].to_dict(orient='records')
sites = sites[['Library_code',  'Host_taxon', 'No_extracts', 'Site_code', 'Habitat', 'Longitude', 'Latitude', 'Location', 'Date']]
sites = sites.rename(columns={'Site_code': 'site', 'Library_code': 'library', 'Habitat': 'habitat', 'No_extracts': 'n_extracts', 'Host_taxon':'host_taxon', 'Longitude': 'longitude', 'Latitude':'latitude', 'Location':'location', 'Date':'date'})
sites = sites.sort_values(by=['site', 'library'])
sites['l'] = sites['library']
sites = sites.set_index('library')
sites = sites.groupby(['l'], as_index=True).transform(lambda x : ";".join([str(m) for m in x.unique()])).reset_index().drop_duplicates(subset=['library'])
sites

,library,host_taxon,n_extracts,site,habitat,longitude,latitude,location,date
0,PV534,Diplotaxis erucoides,3,C1,Crop,-3.593308,40.051302,Aranjuez,1/2/16
1,PV535,Brassica oleracea,17,C1,Crop,-3.593308,40.051302,Aranjuez,1/2/16
2,PV538,Brassica oleracea,8,C1,Crop,-3.593308,40.051302,Aranjuez,1/2/16
3,PV540,Picris echioides,1,C1,Crop,-3.593308,40.051302,Aranjuez,1/2/16
4,PV544,Sisymbrium runcinatum,4,C1,Crop,-3.593308,40.051302,Aranjuez,1/2/16
...,...,...,...,...,...,...,...,...,...
330,PV590,Zea mays,11,Z1,Crop,-3.476076,40.050052,Villaconejos,16/7/15
331,PV047,Zea mays,13,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,24/7/15
332,PV048,Desconocida 4,9,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,24/7/15
333,PV527,Convolvulus arvensis,4,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,24/7/15


In [10]:
db.save_dataframe(
    sites, table_name="d_TableS1", 
    description="Table S1: Library sites and context"
)

Saved d_TableS1 to db.2025-10-27


In [11]:
si.save_dataframe(
    sites, table_name="TableS1", 
    description="Table S1: Library sites and context"
)

Saved TableS1 to si.2025-10-27


In [12]:
si.conn.close()
db.conn.close()